# 1D-CNN: Rowing Stroke Quality Classifier

## Table of Contents
1. [Motivation](#motivation)
2. [Installation](#installation)
3. [Data Loading](#data-loading)
4. [Feature Preparation](#feature-preparation)
5. [Sequence Preparation](#sequence-preparation)
6. [Train/Test Split](#traintest-split)
7. [Model Architecture](#model-architecture)
8. [Training](#training)
9. [Evaluation](#evaluation)
10. [Comparative Analysis](#comparative-analysis)
11. [Discussion](#discussion)


## Motivation

### Why CNN over mean + std aggregation?

Das vorherige Notebook fasst jeden Schlag zu Mittelwert + Streuung zusammen — dabei geht die zeitliche Reihenfolge der Frames verloren. Ein 1D-CNN bekommt stattdessen die **komplette Sequenz** aller Frames eines Schlags und kann Muster darin erkennen:

```
Mittelwert+Std:   [Schlag 1] → 198 Zahlen → Modell

1D-CNN:           [Schlag 1] → Frame 1 → Frame 2 → ... → Frame 28 → Modell
                              ↑ Reihenfolge bleibt erhalten
```

### Wie ein 1D-CNN funktioniert

Ein **Filter** (kleines Fenster) gleitet über die Framesequenz und sucht nach lokalen Mustern:

```
Frames:   [1]  [2]  [3]  [4]  [5]  [6]  ...  [28]

Filter:   [____|____|____]                     ← erkennt Muster in 5 aufeinanderfolgenden Frames
               [____|____|____]
                    [____|____|____]
                         ...
```

Jedes Filter lernt ein anderes Muster — z.B. "typische Ellbogenbewegung in der Zugphase".

Danach fasst **GlobalAveragePooling** alle erkannten Muster zusammen und eine Dense-Schicht entscheidet: GOOD oder BAD.

### Daten-Limitation

Wir haben nur **21 BAD-Schläge**, alle aus einem Video. Ein CNN kann die zeitliche Sequenz besser nutzen als Mittelwert+Std — aber das Confounding-Problem (Modell lernt vielleicht das Video, nicht die Technik) bleibt bestehen. Das wird in der Diskussion adressiert.


## Installation

TensorFlow wird für das Keras-Modell benötigt. Die Zelle unten installiert es einmalig.


In [2]:
# Einmalig ausführen falls TensorFlow noch nicht installiert ist
import importlib, subprocess, sys
if importlib.util.find_spec("tensorflow") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow", "-q"])
    print("TensorFlow installiert. Kernel neu starten falls Fehler auftreten.")
else:
    import tensorflow as tf
    print(f"TensorFlow {tf.__version__} bereits vorhanden.")


ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)
ERROR: No matching distribution found for tensorflow


CalledProcessError: Command '['/usr/local/bin/python3', '-m', 'pip', 'install', 'tensorflow', '-q']' returned non-zero exit status 1.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "requirements.txt").exists()
)
DATA_PATH = PROJECT_ROOT / "1_DatasetCharacteristics" / "landmarks" / "landmarks_all_10fps.csv"
print(f"TensorFlow {tf.__version__}")


## Data Loading


In [ ]:
df = pd.read_csv(DATA_PATH)
feature_cols = [c for c in df.columns if c not in ("Good Stroke", "Bad Stroke")]
X_raw   = df[feature_cols].values
y_frame = df["Good Stroke"].values   # 1 = GOOD, 0 = BAD

print(f"Frames gesamt : {len(X_raw)}")
print(f"GOOD-Frames   : {y_frame.sum()} ({y_frame.mean()*100:.1f}%)")
print(f"BAD-Frames    : {(1-y_frame).sum()} ({(1-y_frame).mean()*100:.1f}%)")


## Feature Preparation

Als Features pro Frame werden die **8 Gelenkwinkel** verwendet (aus dem one_class_classification-Notebook bekannt): Ellbogen, Schulter, Hüfte und Knie links und rechts.

Warum Winkel statt rohe Koordinaten?
- Winkel sind unabhängig davon, wie weit der Ruderer von der Kamera entfernt ist
- 8 aussagekräftige Features statt 99 hochkorrelierter Koordinaten
- Das CNN muss weniger lernen → besser bei kleinen Datensätzen


In [ ]:
# Schlag-IDs zuweisen (gleiche Logik wie in den anderen Notebooks)
VIDEO_INFO = [
    ("cla-BAD",       506,  21),
    ("cla-GOOD-fast", 2094, 88),
    ("cla-GOOD-slow", 1900, 68),
    ("mar-GOOD-fast", 1406, 62),
    ("mar-GOOD-slow", 1359, 51),
]

stroke_ids = []
global_stroke_id = 0
for name, n_frames, n_strokes in VIDEO_INFO:
    frames_per_stroke = n_frames / n_strokes
    for i in range(n_frames):
        stroke_ids.append(global_stroke_id + int(i // frames_per_stroke))
    global_stroke_id += n_strokes

stroke_ids = np.array(stroke_ids)
print(f"Schläge gesamt: {len(np.unique(stroke_ids))}  (21 BAD + 269 GOOD)")


In [ ]:
# Gelenkwinkel berechnen (vektorisiert über alle Frames)
def joint_angle(a, b, c):
    """Winkel in Grad am Punkt b zwischen den Strahlen b→a und b→c."""
    ba = a - b
    bc = c - b
    denom = np.linalg.norm(ba, axis=1) * np.linalg.norm(bc, axis=1) + 1e-9
    cos_a = np.einsum("ij,ij->i", ba, bc) / denom
    return np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0)))

ANGLE_DEFS = [
    ("elbow_left",     "left_wrist",     "left_elbow",     "left_shoulder"),
    ("elbow_right",    "right_wrist",    "right_elbow",    "right_shoulder"),
    ("shoulder_left",  "left_elbow",     "left_shoulder",  "left_hip"),
    ("shoulder_right", "right_elbow",    "right_shoulder", "right_hip"),
    ("hip_left",       "left_shoulder",  "left_hip",       "left_knee"),
    ("hip_right",      "right_shoulder", "right_hip",      "right_knee"),
    ("knee_left",      "left_hip",       "left_knee",      "left_ankle"),
    ("knee_right",     "right_hip",      "right_knee",     "right_ankle"),
]
ANGLE_NAMES = [name for name, *_ in ANGLE_DEFS]
N_ANGLES    = len(ANGLE_NAMES)

df_lm = pd.DataFrame(X_raw, columns=feature_cols)

def get_xyz(lm):
    return df_lm[[f"{lm}_x", f"{lm}_y", f"{lm}_z"]].values

angle_matrix = np.column_stack([
    joint_angle(get_xyz(a), get_xyz(b), get_xyz(c))
    for _, a, b, c in ANGLE_DEFS
])

print(f"Winkel-Matrix: {angle_matrix.shape}  ({angle_matrix.shape[0]} Frames × {angle_matrix.shape[1]} Winkel)")
print(f"Winkel (Frame 0, in Grad): { {n: round(v,1) for n,v in zip(ANGLE_NAMES, angle_matrix[0])} }")


## Sequence Preparation

Das CNN bekommt pro Schlag eine **Sequenz** aller Frames — nicht nur Mittelwert und Streuung.

Da verschiedene Schläge unterschiedlich viele Frames haben (~22–30), werden alle auf eine einheitliche Länge gebracht:
- **Kürzer als MAX_LEN**: mit Nullen aufgefüllt (Padding)
- **Länger als MAX_LEN**: abgeschnitten (Truncation)

```
Schlag A (25 Frames): [f1, f2, ..., f25,  0,  0,  0,  0,  0,  0,  0]  → 32 Frames
Schlag B (28 Frames): [f1, f2, ..., f28,  0,  0,  0,  0]               → 32 Frames
Schlag C (32 Frames): [f1, f2, ..., f32]                                → 32 Frames
```

Ergebnis: ein 3D-Array der Form **(290 Schläge × 32 Frames × 8 Winkel)**.


In [ ]:
# Tatsächliche Framelänge pro Schlag ermitteln
frames_per_stroke = np.bincount(stroke_ids)
print(f"Frames pro Schlag — Min: {frames_per_stroke.min()}  Max: {frames_per_stroke.max()}  Median: {np.median(frames_per_stroke):.0f}")

MAX_LEN = int(frames_per_stroke.max()) + 2   # +2 als Puffer
print(f"→ MAX_LEN = {MAX_LEN}")


In [ ]:
# Sequenzen aufbauen: (n_strokes, MAX_LEN, N_ANGLES)
unique_strokes = np.unique(stroke_ids)
n_strokes = len(unique_strokes)

X_seq = np.zeros((n_strokes, MAX_LEN, N_ANGLES), dtype=np.float32)
y_seq = np.zeros(n_strokes, dtype=np.float32)

for i, sid in enumerate(unique_strokes):
    mask   = stroke_ids == sid
    frames = angle_matrix[mask]          # (n_frames, 8)
    label  = y_frame[mask][0]            # alle Frames eines Schlags haben dasselbe Label
    n      = min(len(frames), MAX_LEN)
    X_seq[i, :n, :] = frames[:n]
    y_seq[i]         = label

print(f"Sequenz-Array: {X_seq.shape}")
print(f"  Bedeutung  : ({n_strokes} Schläge × {MAX_LEN} Frames × {N_ANGLES} Winkel)")
print(f"GOOD-Schläge : {int(y_seq.sum())}")
print(f"BAD-Schläge  : {int((1-y_seq).sum())}")


## Train/Test Split

Wir teilen **schlagweise** auf (nicht frameweise), damit kein Schlag halb im Training und halb im Test landet.

`StratifiedShuffleSplit` stellt sicher, dass beide Splits die gleiche GOOD/BAD-Verteilung haben — wichtig weil wir nur 21 BAD-Schläge haben.

**Hinweis zur Klassen-Ungleichverteilung:** Das Modell würde ohne Gegenmaßnahme einfach alles als GOOD vorhersagen (93% Accuracy, 0% BAD Recall). Daher werden **Klassengewichte** gesetzt: BAD-Schläge werden stärker gewichtet, als wären es mehr Beispiele.


In [ ]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(sss.split(X_seq, y_seq))

X_train, X_test = X_seq[train_idx], X_seq[test_idx]
y_train, y_test = y_seq[train_idx], y_seq[test_idx]

print(f"Training : {len(X_train)} Schläge  (GOOD: {int(y_train.sum())}  BAD: {int((1-y_train).sum())})")
print(f"Test     : {len(X_test)} Schläge  (GOOD: {int(y_test.sum())}  BAD: {int((1-y_test).sum())})")

# Klassengewichte: gleicht die 93/7-Ungleichverteilung aus
cw = compute_class_weight("balanced", classes=np.array([0.0, 1.0]), y=y_train)
class_weights = {0: cw[0], 1: cw[1]}
print(f"\nKlassengewichte: BAD={cw[0]:.2f}  GOOD={cw[1]:.2f}")
print(f"→ BAD-Schläge werden {cw[0]/cw[1]:.1f}× stärker gewichtet als GOOD-Schläge")


In [ ]:
# Normalisierung: StandardScaler pro Feature (Winkel) über alle Frames und Schläge
# Reshape nötig da StandardScaler 2D erwartet
n_train, t, f = X_train.shape
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train.reshape(-1, f)).reshape(n_train, t, f)
X_test_sc  = scaler.transform(X_test.reshape(-1, f)).reshape(len(X_test), t, f)


## Model Architecture

```
Eingabe: (32 Frames × 8 Winkel)
        ↓
┌───────────────────────────────────┐
│  Conv1D(32 Filter, Fenstergröße 5)│  ← erkennt kurze lokale Muster (z.B. Einzugsphase)
│  BatchNormalization               │
└───────────────────────────────────┘
        ↓
┌───────────────────────────────────┐
│  Conv1D(64 Filter, Fenstergröße 3)│  ← kombiniert lokale Muster zu längeren Strukturen
│  BatchNormalization               │
└───────────────────────────────────┘
        ↓
┌───────────────────────────────────┐
│  GlobalAveragePooling1D           │  ← fasst den ganzen Schlag zu einem Vektor zusammen
└───────────────────────────────────┘
        ↓
┌───────────────────────────────────┐
│  Dense(32)  +  Dropout(0.4)       │  ← Klassifikationskopf
└───────────────────────────────────┘
        ↓
┌───────────────────────────────────┐
│  Dense(1, sigmoid)                │  ← Ausgabe: Wahrscheinlichkeit für GOOD (0–1)
└───────────────────────────────────┘
```

**Dropout** schaltet beim Training zufällig 40% der Neuronen aus — das verhindert Auswendiglernen bei einem kleinen Datensatz.

**BatchNormalization** stabilisiert das Training indem sie die Aktivierungen normalisiert.


In [ ]:
def build_1d_cnn(max_len, n_features):
    inputs = keras.Input(shape=(max_len, n_features), name="stroke_sequence")

    x = layers.Conv1D(32, kernel_size=5, padding="same", activation="relu", name="conv1")(inputs)
    x = layers.BatchNormalization(name="bn1")(x)

    x = layers.Conv1D(64, kernel_size=3, padding="same", activation="relu", name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)

    x = layers.GlobalAveragePooling1D(name="global_pool")(x)

    x = layers.Dense(32, activation="relu", name="dense")(x)
    x = layers.Dropout(0.4, name="dropout")(x)

    output = layers.Dense(1, activation="sigmoid", name="output")(x)

    return keras.Model(inputs, output, name="1D_CNN_Stroke_Classifier")


model = build_1d_cnn(MAX_LEN, N_ANGLES)
model.summary()


## Training

**EarlyStopping** beobachtet den Validierungsverlust: wenn er sich 20 Epochen lang nicht verbessert, wird das Training gestoppt und die beste Version des Modells wiederhergestellt. Das verhindert Overfitting.


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train_sc, y_train,
    epochs=150,
    batch_size=16,
    validation_split=0.15,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
# Trainingskurven
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, title in zip(
    axes,
    ["loss", "accuracy"],
    ["Loss (Binary Crossentropy)", "Accuracy"]
):
    ax.plot(history.history[metric],          label="Training",    color="steelblue")
    ax.plot(history.history[f"val_{metric}"], label="Validierung", color="darkorange")
    ax.set_xlabel("Epoche")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Trainingsverlauf 1D-CNN", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Training gestoppt nach Epoche {len(history.history['loss'])}.")


## Evaluation


In [ ]:
y_proba = model.predict(X_test_sc, verbose=0).flatten()
y_pred  = (y_proba >= 0.5).astype(int)

auc    = roc_auc_score(y_test, y_proba)
f1_mac = f1_score(y_test, y_pred, average="macro")
report = classification_report(y_test, y_pred, target_names=["BAD", "GOOD"], output_dict=True)

print(f"{'='*54}")
print(f"  1D-CNN Evaluation (Test Set)")
print(f"{'='*54}")
print(f"  ROC-AUC     : {auc:.4f}")
print(f"  Macro F1    : {f1_mac:.4f}")
print(f"  BAD Recall  : {report['BAD']['recall']:.4f}  ← % der BAD-Schläge erkannt")
print(f"  BAD Prec.   : {report['BAD']['precision']:.4f}  ← % der Alarme wirklich BAD")
print()
print(classification_report(y_test, y_pred, target_names=["BAD", "GOOD"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=["BAD", "GOOD"]
).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix — 1D-CNN")

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_proba, name="1D-CNN", ax=axes[1], color="steelblue")
axes[1].plot([0, 1], [0, 1], "k--", label="Zufall")
axes[1].set_title("ROC-Kurve — 1D-CNN")
axes[1].legend()

plt.suptitle("1D-CNN Ergebnisse (Test Set)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Vorhersage-Wahrscheinlichkeiten je Klasse visualisieren
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_proba[y_test == 1], bins=15, alpha=0.65, color="steelblue", label="GOOD-Schläge", density=True)
ax.hist(y_proba[y_test == 0], bins=10, alpha=0.65, color="crimson",   label="BAD-Schläge",  density=True)
ax.axvline(0.5, color="black", linestyle="--", label="Entscheidungsgrenze (0.5)")
ax.set_xlabel("Vorhergesagte Wahrscheinlichkeit für GOOD")
ax.set_ylabel("Dichte")
ax.set_title("Vorhersageverteilung nach Klasse")
ax.legend()
plt.tight_layout()
plt.show()


## Comparative Analysis

Vergleich mit den Modellen aus den vorherigen Notebooks (auf Schlagebene).

**Wichtig:** Die Test-Sets sind nicht identisch:
- 1D-CNN und RF: stratifizierter Split (~20% aller Schläge, ~4 BAD im Test)
- One-Class-Modelle: alle 21 BAD-Schläge im Test

Direkte Zahlenvergleiche sind deshalb nur bedingt aussagekräftig.


In [ ]:
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM

# --- Gemeinsame Schlag-Features (Mittelwert+Std der Winkel) für RF und One-Class ---
df_ang = pd.DataFrame(angle_matrix, columns=ANGLE_NAMES)
df_ang["stroke_id"] = stroke_ids

stroke_means = df_ang.groupby("stroke_id")[ANGLE_NAMES].mean()
stroke_stds  = df_ang.groupby("stroke_id")[ANGLE_NAMES].std().fillna(0)
stroke_stds.columns = [f"{n}_std" for n in ANGLE_NAMES]
X_flat   = np.hstack([stroke_means.values, stroke_stds.values])
y_flat   = np.array([y_frame[stroke_ids == sid][0] for sid in np.unique(stroke_ids)])

# Gleicher Split wie beim CNN für Vergleichbarkeit
X_flat_train, X_flat_test = X_flat[train_idx], X_flat[test_idx]
y_flat_train, y_flat_test = y_flat[train_idx], y_flat[test_idx]

scaler_flat  = StandardScaler()
Xf_train_sc  = scaler_flat.fit_transform(X_flat_train)
Xf_test_sc   = scaler_flat.transform(X_flat_test)

# Random Forest (supervised, mit BAD im Training)
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(Xf_train_sc, y_flat_train)
rf_proba = rf.predict_proba(Xf_test_sc)[:, 1]
rf_pred  = rf.predict(Xf_test_sc)

# Isolation Forest (one-class, nur GOOD im Training)
X_good_train_flat = Xf_train_sc[y_flat_train == 1]
iso = IsolationForest(n_estimators=200, contamination=0.1, random_state=RANDOM_STATE, n_jobs=-1)
iso.fit(X_good_train_flat)
iso_scores = iso.score_samples(Xf_test_sc)

print(f"Test-Set: {len(y_flat_test)} Schläge — GOOD: {int(y_flat_test.sum())}  BAD: {int((1-y_flat_test).sum())}")


In [ ]:
def metrics(y_true, y_score_or_pred, is_score=True):
    if is_score:
        auc  = roc_auc_score(y_true, y_score_or_pred)
        pred = (y_score_or_pred >= 0.5).astype(int)
    else:
        # one-class: score already computed
        auc  = roc_auc_score(y_true, y_score_or_pred)
        pred = (y_score_or_pred >= np.percentile(y_score_or_pred, 10)).astype(int)
    rep  = classification_report(y_true, pred, target_names=["BAD","GOOD"], output_dict=True, zero_division=0)
    f1m  = f1_score(y_true, pred, average="macro", zero_division=0)
    return auc, f1m, rep["BAD"]["recall"]

cnn_auc, cnn_f1, cnn_rec = metrics(y_test,      y_proba)
rf_auc,  rf_f1,  rf_rec  = metrics(y_flat_test, rf_proba)
iso_auc, iso_f1, iso_rec = metrics(y_flat_test, iso_scores, is_score=False)

print(f"{'Modell':<30} {'ROC-AUC':>8} {'Macro F1':>9} {'BAD Recall':>11} {'BAD im Training':>16}")
print("-" * 79)
print(f"  {'1D-CNN':<28} {cnn_auc:>8.4f} {cnn_f1:>9.4f} {cnn_rec:>11.4f} {'Ja (wenige)':>16}")
print(f"  {'Random Forest':<28} {rf_auc:>8.4f} {rf_f1:>9.4f} {rf_rec:>11.4f} {'Ja':>16}")
print(f"  {'Isolation Forest':<28} {iso_auc:>8.4f} {iso_f1:>9.4f} {iso_rec:>11.4f} {'Nein':>16}")
print()
print("Alle Modelle verwenden denselben Test-Split für diesen Vergleich.")


## Discussion

### Was das 1D-CNN besser macht

Das CNN bekommt die **zeitliche Abfolge** aller Frames eines Schlags — nicht nur Mittelwert und Streuung. Es kann dadurch lernen, was in der Einzugsphase, der Zugphase und der Rückholphase jeweils passiert, und diese Phasen unterschiedlich gewichten.

### Einschränkungen

**Zu wenig Daten:** Mit nur 290 Schlägen (21 davon BAD) ist ein CNN stark overfitting-gefährdet. Das CNN hat tausende Parameter — mehr als die Anzahl der Trainingsbeispiele. Dropout und EarlyStopping mildern das, lösen es aber nicht vollständig.

**Data Confounding bleibt:** Alle BAD-Schläge kommen aus einem einzigen Video. Das CNN kann trotz der zeitlichen Sequenz immer noch lernen, wie das `cla-BAD`-Video aussieht — nicht wie schlechte Technik aussieht. Auf einem neuen Ruderer mit schlechter Technik würde es wahrscheinlich versagen.

**Vergleich der Ansätze:**

| Ansatz | Stärke | Schwäche |
|--------|--------|---------|
| Mittelwert+Std → RF | Einfach, interpretierbar | Verliert Zeitinformation |
| One-Class (IF/SVM) | Kein BAD nötig, generalisierbar | Weniger diskriminativ |
| 1D-CNN | Nutzt Zeitstruktur, ausdrucksstark | Braucht mehr Daten, Overfitting-Risiko |

### Fazit

Für den vorliegenden Datensatz (290 Schläge, 21 BAD aus einem Video) ist das 1D-CNN keine klare Verbesserung über die einfacheren Ansätze — der Vorteil der Zeitstruktur wird durch die geringe Datenmenge und das Confounding-Problem aufgewogen. Mit mehr Daten (mehrere Ruderer, mehrere BAD-Videos) würde ein 1D-CNN deutlich profitieren.
